# RFM + CLV

Builds RFM features and predictive 12-month CLV (BG/NBD + Gamma-Gamma) per customer.

**Note:** `lifetimes`' internal `recency`/`T` fields are NOT the same as everyday
"days since last purchase" — they're used only to fit the CLV model. A separate,
correctly-defined `days_since_last_purchase` field is built here for any
downstream business use.

In [1]:
import pandas as pd
transactions = pd.read_csv('../data/processed/transactions.csv', parse_dates=['order_purchase_timestamp'])
transactions.head()

,customer_unique_id,order_id,order_purchase_timestamp,order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,27.19
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,43.62
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,196.89


In [5]:
snapshot_date = transactions['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = transactions.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('order_value', 'sum')
).reset_index()

rfm.describe()

C:\Users\bumba\AppData\Local\Temp\ipykernel_16188\4246702924.py:1: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  snapshot_date = transactions['order_purchase_timestamp'].max() + pd.Timedelta(days=1)


,recency,frequency,monetary
count,93357.000000,93357.000000,93357.000000
mean,237.936673,1.033420,211.833718
std,152.584315,0.209099,642.166523
min,1.000000,1.000000,9.590000
25%,114.000000,1.000000,63.760000
50%,219.000000,1.000000,112.950000
75%,346.000000,1.000000,201.740000
max,695.000000,15.000000,109312.640000


In [4]:
print((rfm['frequency'] > 1).sum(), "repeat customers out of", len(rfm))

2801 repeat customers out of 93357


In [6]:
pip install lifetimes

Note: you may need to restart the kernel to use updated packages.


In [7]:
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

summary = summary_data_from_transaction_data(
    transactions,
    'customer_unique_id',
    'order_purchase_timestamp',
    monetary_value_col='order_value',
    observation_period_end=snapshot_date
)

summary_repeat = summary[summary['frequency'] > 0]
print(f"Repeat customers for Gamma-Gamma fit: {len(summary_repeat)}")

bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(summary_repeat['frequency'], summary_repeat['monetary_value'])

print(bgf)
print(ggf)

Repeat customers for Gamma-Gamma fit: 2015
<lifetimes.BetaGeoFitter: fitted with 93357 subjects, a: 0.35, alpha: 73.67, b: 0.06, r: 0.02>
<lifetimes.GammaGammaFitter: fitted with 2015 subjects, p: 9.67, q: 1.12, v: 9.00>


In [8]:
summary['predicted_clv_12m'] = ggf.customer_lifetime_value(
    bgf,
    summary['frequency'],
    summary['recency'],
    summary['T'],
    summary['monetary_value'],
    time=12,
    freq='D',
    discount_rate=0.01
)

summary = summary.reset_index().rename(columns={'index': 'customer_unique_id'})
summary['predicted_clv_12m'].describe()

count    93357.000000
mean        11.079943
std         20.850231
min          0.263020
25%          7.141577
50%          9.443173
75%         12.962604
max       3937.108328
Name: predicted_clv_12m, dtype: float64

In [9]:
summary['clv_tier'] = pd.qcut(summary['predicted_clv_12m'], q=4, labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

summary.groupby('clv_tier')['predicted_clv_12m'].agg(['count', 'mean', 'sum'])

,count,mean,sum
clv_tier,,,
Bronze,23474,5.973143,140213.551224
Silver,23251,8.289375,192736.257494
Gold,23438,11.035134,258641.469597
Platinum,23194,19.091101,442799.006371


In [10]:
summary.to_csv('../data/processed/customer_clv.csv', index=False)